# 🔁 Replication & Partitioning (Sharding) — Complete Guide

> w Definitier:**
> *"Replication is keeping copies of the same data on multiple machines — for availability and fault tolerance. Partitioning (Sharding) is splitting data across multiple machines — for scalability and throughput. Real systems use both together: partition data to scale writes, replicate each partition to survive failures."*

---

## Table of Contents

### Replication
1. [Why Replication is Needed](#1-why-replication-is-needed)
2. [Single Leader Replication](#2-single-leader-replication)
3. [Multi Leader Replication](#3-multi-leader-replication)
4. [Leaderless Replication](#4-leaderless-replication)
5. [Sync vs Async Replication](#5-sync-vs-async-replication)
6. [Quorum](#6-quorum)

### Partitioning (Sharding)
7. [Why Partitioning is Needed](#7-why-partitioning-is-needed)
8. [Partition by Key (Range Partitioning)](#8-partition-by-key-range-partitioning)
9. [Partition by Hash](#9-partition-by-hash)
10. [Secondary Index Partitioning](#10-secondary-index-partitioning)
11. [Hotspots and Challenges](#11-hotspots-and-challenges)
12. [Replication vs Partitioning vs Both](#12-replication-vs-partitioning-vs-both)
13. [Quick Revision Cheatsheet](#13-quick-revision-cheatsheet)

---

# PART 1 — REPLICATION

---

## 1. Why Replication is Needed

### 🧠 Simple Definition
Replication means **keeping identical copies of your data on multiple machines** (called replicas or nodes) simultaneously. Every write that happens on the primary is mirrored to one or more secondary machines.

> **Interview Line to Remember:**
> *"Replication solves two things: availability (if one machine dies, others have the data) and read scalability (spread read traffic across replicas). The core challenge of replication is keeping all copies in sync when data changes."*

---

### ❓ What Problem Does It Solve?

**Problem 1 — Single Point of Failure:**
One database server holds all your data. It crashes → your entire application loses data access → full outage. Replication gives you backup copies on standby.

**Problem 2 — Read Scalability:**
One DB server handles all reads and writes. At scale, reads vastly outnumber writes (read-heavy workloads: 90% reads, 10% writes). Replicas can serve reads while the leader handles writes — distributing the read load.

**Problem 3 — Geographic Latency:**
Users in India querying a database in the US experience 200ms latency per query. A replica in India brings that to 5ms.

### 🏭 Why Needed in Production?
- **Zero data loss on server failure** — replica immediately takes over (failover)
- **Read scaling** — route read-heavy traffic (dashboards, analytics, reports) to replicas
- **Disaster recovery** — replica in different availability zone/region survives datacenter outage
- **Zero-downtime maintenance** — take primary offline for maintenance; replica serves traffic
- **Compliance** — some regulations require data to exist in multiple physical locations

### 💥 What Happens Without It?
- Single DB crashes at 3 AM → entire app down → data inaccessible until server restored
- Traffic grows → single DB overwhelmed by reads → query latency spikes → app slows
- Datacenter fire or flood → one copy of data → data permanently lost → company-ending event

### 🏢 Real Company Example
**GitHub** uses MySQL with replication. Their primary MySQL server takes all writes. Multiple read replicas serve the millions of read queries (viewing repos, commits, profiles). When their primary had an incident in 2018, replicas kept reads running — zero data loss.

---

## 2. Single Leader Replication

### 🧠 Simple Definition
Single Leader (also called **Master-Slave** or **Primary-Replica**) replication designates **one node as the leader (primary)**. All writes go exclusively to the leader. The leader replicates changes to follower nodes. Reads can go to the leader or any follower.

> **Interview Line to Remember:**
> *"Single Leader = one writer, many readers. All writes go to the primary. Changes flow downstream to replicas. Simple, consistent, and the most widely used replication model. The tradeoff: the leader is still a single point of failure for writes."*

---

### ❓ What Problem Does It Solve?
The core challenge of replication is **write conflicts** — what happens if two nodes accept different writes at the same time? Single leader eliminates this entirely by funneling all writes through one node. There is only one source of truth for writes — no conflicts possible.

### 🏭 Why Needed in Production?
- **Simplest consistency model** — one leader = no write conflicts = easy to reason about
- **Read scaling** — add as many read replicas as needed; route reporting/analytics there
- **Automatic failover** — if leader dies, a replica is promoted to leader (seconds of downtime)
- **Default in most databases** — PostgreSQL, MySQL, MongoDB all use this by default
- **Replication lag monitoring** — easy to measure how far behind each replica is

### 💥 What Happens Without A Clear Leader?
Multiple nodes accepting writes simultaneously → **write conflicts** → "which write wins?" → data inconsistency → corrupted data. Single leader sidesteps this entirely.

### 💥 Single Leader's Own Weakness:
- **Leader is write bottleneck** — all writes must go through one machine
- **Replication lag** — replicas may be seconds behind the leader (stale reads)
- **Leader failure** = temporary write unavailability during failover

### 🏢 Real Company Example
**PostgreSQL Streaming Replication** — used by almost every company using Postgres (Airbnb, Instagram, Notion). Primary server takes all writes. One or more standbys stream changes in real time. Read replicas serve analytics. If primary dies, a standby is promoted using `pg_promote()` — automated by tools like Patroni at Uber.

---

### 📐 Single Leader Architecture

```
                         ┌─────────────────────────┐
All Writes ─────────────►│   PRIMARY (Leader)       │
                         │   - Accepts all writes   │
                         │   - Source of truth      │
                         └───────────┬─────────────┘
                                     │
                    Replication stream (changes flow down)
                    ┌────────────────┼────────────────┐
                    ▼                ▼                ▼
             ┌──────────┐    ┌──────────┐    ┌──────────┐
             │ Replica 1│    │ Replica 2│    │ Replica 3│
             │ (reads)  │    │ (reads)  │    │ (standby)│
             └──────────┘    └──────────┘    └──────────┘
                    ▲                ▲
              Read traffic      Read traffic
              (reports,         (API reads)
               dashboards)

Failover: Primary dies → Replica 3 promoted to Primary automatically
```

---

### 📐 Replication Lag — The Hidden Problem

```
Timeline:
  t=0:   User writes  "email = new@email.com"  → Primary
  t=0:   Primary confirms write success → User sees success
  t=1s:  Replica 1 receives and applies the change
  t=2s:  Replica 2 receives and applies the change

  If user reads from Replica 1 at t=0.5s:
  → Gets old email (change not yet arrived) ← STALE READ
  → This is called Replication Lag

Solutions:
  → Read-your-own-writes: after a write, read from primary for 1-2 seconds
  → Monotonic reads: user always reads from same replica
  → Bounded staleness: application tolerates max N seconds of lag
```

---

## 3. Multi Leader Replication

### 🧠 Simple Definition
Multi Leader replication (also called **Master-Master** or **Active-Active**) allows **multiple nodes to accept writes simultaneously**. Each leader replicates its changes to all other leaders. No single bottleneck for writes.

> **Interview Line to Remember:**
> *"Multi Leader = multiple writers. Each datacenter has its own leader that accepts local writes. Leaders sync with each other. Solves cross-datacenter write latency. The hard problem: write conflicts — what if two leaders update the same record at the same time?"*

---

### ❓ What Problem Does It Solve?
In Single Leader across datacenters, all writes must cross the ocean to the primary leader — a Mumbai user's write goes to the US primary and back, adding 200ms latency per write. Multi Leader puts a leader in each datacenter — Mumbai users write to the Mumbai leader locally (5ms), which then syncs to other regions asynchronously.

### 🏭 Why Needed in Production?
- **Multi-datacenter writes** — users in each region write to local leader; no cross-ocean write latency
- **Offline-capable apps** — each device is its own "leader" (Google Docs offline, CRDTs)
- **High write throughput** — multiple leaders share write load; no single write bottleneck
- **Continued operation during network partition** — each datacenter works independently if connection between them breaks

### 💥 What Happens Without It?
Single leader in US East — Indian office users see 200-400ms latency on every write → forms feel slow → user experience degrades → complaints → lost business.

### 💥 Multi Leader's Hard Problem — Write Conflicts:
```
User A (Mumbai leader):  UPDATE product SET price = 999
User B (US leader):      UPDATE product SET price = 1299
(Both happen at the same time, both leaders accept the write)

When leaders sync: which price wins? 999 or 1299?
→ This is a write conflict. Must be resolved.
```

**Conflict Resolution Strategies:**
| Strategy | How | Used By |
|---|---|---|
| **Last Write Wins (LWW)** | Timestamp decides — latest write wins | Cassandra (default) |
| **Higher Node ID wins** | Deterministic but arbitrary | Some configs |
| **Application-level merge** | App defines custom logic | Google Docs (CRDT) |
| **Show conflict to user** | User manually resolves | CouchDB |

### 🏢 Real Company Example
**Google Docs** — multi-leader at the device level. You can edit offline on your laptop, phone edits online simultaneously. Both are "leaders." When you reconnect, Google uses **CRDTs (Conflict-free Replicated Data Types)** to merge changes mathematically without conflicts. Your offline edits and your colleague's online edits are both preserved.

---

### 📐 Multi Leader Architecture

```
         Mumbai DC                           US DC
    ┌─────────────────┐               ┌─────────────────┐
    │  Mumbai Leader  │◄─────sync────►│   US Leader     │
    │  Accepts writes │               │  Accepts writes  │
    │  from IN users  │               │  from US users   │
    └────────┬────────┘               └────────┬────────┘
             │                                 │
     Mumbai Replicas                     US Replicas
    ┌────────┴────────┐               ┌────────┴────────┐
    │ R1  │  R2  │ R3 │               │ R1  │  R2  │ R3 │
    └─────────────────┘               └─────────────────┘

Mumbai user write: Mumbai Leader (5ms) ✅  — no cross-ocean trip
US user write:     US Leader (5ms) ✅      — no cross-ocean trip
Sync between leaders: async, eventually consistent
```

---

## 4. Leaderless Replication

### 🧠 Simple Definition
Leaderless replication has **no designated leader at all**. Clients send writes to **multiple nodes simultaneously**. A write is considered successful when a **quorum** (minimum number) of nodes acknowledge it. Reads also go to multiple nodes, and the freshest value wins.

> **Interview Line to Remember:**
> *"Leaderless = no single master. Client writes to multiple nodes at once and waits for a quorum of acknowledgements. Client reads from multiple nodes and takes the most up-to-date value. Highest availability — any node can serve any request. Used by Cassandra, DynamoDB, Riak."*

---

### ❓ What Problem Does It Solve?
Both Single and Multi Leader have a leadership concept — if the leader is unavailable, something must happen (failover, re-election). Leaderless eliminates this entirely. There's no leader to fail. Any node can take any request. The system keeps working even if multiple nodes are down simultaneously.

### 🏭 Why Needed in Production?
- **Extreme availability** — no leader election, no failover delay; works with N nodes down
- **Any node can serve reads/writes** — no "leader" to route to, no routing table needed
- **Massively scalable writes** — writes spread across all nodes, no bottleneck
- **Tunable consistency** — choose between strong consistency (higher quorum) and high availability (lower quorum) per request

### 💥 What Happens Without It?
With leader-based systems during leader failure: 10–30 seconds of unavailability during leader election. For systems requiring 99.999% availability (5 nines), even 10 seconds of monthly downtime is too much. Leaderless gives near-zero downtime.

### 🏢 Real Company Example
**Apache Cassandra** (used by Netflix, Instagram, Apple) — Cassandra has no master. Every node is equal. A write to Netflix's viewing history goes to multiple Cassandra nodes simultaneously. Netflix can take down any subset of Cassandra nodes for maintenance; the cluster keeps serving reads and writes from the remaining nodes.

---

### 📐 Leaderless Architecture

```
Leaderless — All nodes are equal peers:

         Client
           │
    Write to all simultaneously
    ┌──────┼──────┐
    ▼      ▼      ▼
  [N1]   [N2]   [N3]   [N4]   [N5]
   ✅     ✅     ✅    (down)  (down)

Quorum = 3 nodes must ACK → 3/5 responded ✅ → Write successful
N4, N5 are down — doesn't matter, quorum still met

Read — ask multiple nodes, take freshest:
    Client reads from N1, N2, N3
    N1 → version 5 (stale, missed a write)
    N2 → version 7 (latest) ← use this
    N3 → version 7 (latest) ← matches
    Client returns version 7 ✅
    Background: client repairs N1 (Read Repair)
```

---

### 📐 Read Repair — How Stale Nodes Catch Up

```
Read Repair (Cassandra's self-healing mechanism):

Client reads from 3 nodes:
  N1 → {price: 999,  version: 3}  ← stale
  N2 → {price: 1299, version: 5}  ← fresh
  N3 → {price: 1299, version: 5}  ← fresh

Client returns 1299 (latest version) to user.

Background: Client sends version 5 to N1 (repairs it)
Next read from N1 → returns 1299 ✅

This is "eventual consistency" — N1 was stale briefly, then healed.
```

---

## 5. Sync vs Async Replication

### 🧠 Simple Definition
- **Synchronous Replication** — Leader waits for **all (or specific) replicas to confirm** they've received and applied the write before sending success to the client. Slow but no data loss.
- **Asynchronous Replication** — Leader sends write to replicas **in the background** and immediately confirms success to the client. Fast but potential data loss if leader crashes before replicas catch up.

> **Interview Line to Remember:**
> *"Sync replication: leader waits for replica ACK — zero data loss, slower writes. Async replication: leader confirms immediately, replicates later — fast writes, risk of losing recent writes if leader crashes. Production uses semi-sync: wait for ONE replica, replicate to rest async — best of both."*

---

### ❓ What Problem Does It Solve?
The fundamental tension in distributed systems: **durability vs performance**.
- Want zero data loss? Wait for confirmation from replicas → adds latency
- Want fast writes? Confirm immediately → risk losing data if leader crashes before syncing

### 🏭 Why Needed in Production?
- **Financial transactions** — sync replication ensures a confirmed payment is on at least 2 nodes. Never lost.
- **Analytics / social features** — async is fine for likes, views, non-critical data. Speed matters more than perfect durability.
- **Semi-sync** (MySQL default) — wait for 1 replica to ACK, replicate to rest async. Balance of safety and speed.

### 💥 What Happens Without Understanding This?
- Use async for bank transfers → leader crashes after confirm but before replica syncs → ₹10,000 debited but no record on replica → money "disappears" → regulatory disaster
- Use sync for all 5 replicas → write latency = slowest replica's latency → even one slow replica stalls all writes → production performance disaster

### 🏢 Real Company Example
**Google Spanner** — uses **synchronous replication across datacenters** (with Paxos consensus). Every write is confirmed only after a majority of datacenters acknowledge it. Zero data loss, globally consistent. This is why Google Cloud's database SLA can promise 99.999% durability — every write is on multiple continents before being confirmed.

---

### 📐 Sync vs Async vs Semi-Sync

```
SYNCHRONOUS:
  Client ──► Leader ──► Replica 1 (wait for ACK) ──► ACK
                    ──► Replica 2 (wait for ACK) ──► ACK
  Leader sends "success" to Client only after ALL replicas confirm.

  Durability:  ████████████ Maximum (zero data loss)
  Write Speed: ████░░░░░░░░ Slow (waits for slowest replica)
  Use Case: Banking, payments, critical financial data

─────────────────────────────────────────────────────

ASYNCHRONOUS:
  Client ──► Leader ──► "success" ← (immediately confirmed)
  Leader replicates to replicas in background (no waiting)

  Durability:  ████░░░░░░░░ Risk of loss (if leader crashes before sync)
  Write Speed: ████████████ Maximum (no waiting)
  Use Case: Social feeds, analytics, non-critical counters

─────────────────────────────────────────────────────

SEMI-SYNCHRONOUS (Production Default):
  Client ──► Leader ──► Replica 1 (wait for ACK) ──► ACK
                    ──► Replica 2 (async, no wait)
                    ──► Replica 3 (async, no wait)
  Leader sends "success" after ONE replica confirms.

  Durability:  ████████░░░░ Good (at least 2 copies before confirm)
  Write Speed: ████████░░░░ Good (waits for only fastest replica)
  Use Case: Most production databases — MySQL semi-sync, PostgreSQL sync_standby
```

---

## 6. Quorum

### 🧠 Simple Definition
A Quorum is the **minimum number of nodes that must agree** for a read or write operation to be considered successful in a distributed system. It's the voting majority that ensures consistency despite node failures.

The rule: **W + R > N** (where N = total nodes, W = write quorum, R = read quorum)

> **Interview Line to Remember:**
> *"Quorum is the minimum votes needed for a decision. In a 5-node cluster: write to 3 nodes (W=3), read from 3 nodes (R=3). W+R > N ensures the read set and write set always overlap — you're guaranteed to read at least one node that has the latest write. It's the math that makes distributed consistency work."*

---

### ❓ What Problem Does It Solve?
In a distributed system, some nodes might be down or slow. Quorum defines **how many nodes must participate** for an operation to be valid — balancing between:
- Requiring ALL nodes (maximum consistency, zero tolerance for failures)
- Requiring just ONE node (maximum availability, no consistency guarantee)
- Requiring MAJORITY (the sweet spot — tolerates failures, guarantees overlap)

### 🏭 Why Needed in Production?
- **Tolerate node failures** — N=5, W=3: system works even with 2 nodes down
- **Guarantee consistency** — W+R>N means read and write sets always overlap
- **Tunable per operation** — use strong quorum for critical reads, weak for fast reads
- **Foundation of Cassandra, DynamoDB, Riak, ZooKeeper, etcd** — all quorum-based

### 💥 What Happens Without It?
- No quorum: write to 1 node, read from different 1 node → read might miss the write → stale data
- Require all N nodes: one node down → entire cluster unavailable for writes → no fault tolerance

### 🏢 Real Company Example
**Amazon DynamoDB** — default quorum is W=2, R=2, N=3. Every write is confirmed by 2 of 3 nodes. Every read queries 2 of 3 nodes. W+R=4 > N=3 — guaranteed overlap. One DynamoDB node can be down with zero impact on consistency. This is the foundation of DynamoDB's 99.999% availability SLA.

---

### 📐 Quorum Math — The Core Formula

```
N = Total number of replica nodes
W = Write quorum (minimum nodes that must confirm a write)
R = Read quorum (minimum nodes that must respond to a read)

Rule: W + R > N  →  read set and write set ALWAYS overlap
      W + R ≤ N  →  no guarantee of overlap → stale reads possible

Example: N = 5 nodes
─────────────────────────────────────────────────────────
Strong Consistency:  W=3, R=3  →  W+R=6 > 5 ✅
                     Tolerates: 2 node failures for writes AND reads
                     Overlap: at least 1 node in both sets

High Write Speed:    W=1, R=5  →  W+R=6 > 5 ✅
                     Write fast (1 node) but read all 5 (slow reads)

High Read Speed:     W=5, R=1  →  W+R=6 > 5 ✅
                     Write all 5 (slow writes) but read 1 (fast reads)

Eventual Consistency: W=1, R=1 → W+R=2 ≤ 5 ❌ (no overlap guarantee)
                     Maximum speed, no consistency guarantee

Production Sweet Spot: W=2, R=2, N=3  (DynamoDB default)
                        W+R=4 > 3 ✅  Tolerates 1 node failure
```

---

### 📐 Why W + R > N Guarantees Overlap

```
N=5 nodes: [N1] [N2] [N3] [N4] [N5]
W=3: Write goes to N1, N2, N3  (any 3)
R=3: Read  goes to N3, N4, N5  (any 3)

Overlap: N3 is in BOTH write set AND read set
→ N3 has the latest write
→ Reading from N3 returns the fresh data ✅

Even in the worst case, W+R > N guarantees at least 1 overlap node.
That node always has the latest write.
```

---

### 📐 Quorum in Leader Election (Raft/Paxos)

```
Raft Consensus (used by etcd, CockroachDB, Consul):

5-node cluster. Leader dies.
Remaining 4 nodes must elect a new leader.
Quorum needed: majority = floor(5/2) + 1 = 3 nodes must agree.

Node 2 requests votes:
  Node 3: "Yes" ✅
  Node 4: "Yes" ✅
  Node 5: "Yes" ✅  ← 3/5 = quorum reached → Node 2 becomes leader

This prevents "split-brain" — two nodes can't both win majority
in a correctly configured cluster.
```

---

# PART 2 — PARTITIONING (SHARDING)

---

## 7. Why Partitioning is Needed

### 🧠 Simple Definition
Partitioning (also called **Sharding**) is **splitting a large dataset across multiple machines** where each machine holds only a subset of the data. Instead of one database with 10 billion rows, you have 10 shards each with 1 billion rows — distributed across 10 servers.

> **Interview Line to Remember:**
> *"Sharding splits data horizontally across multiple machines — each shard holds a subset of the rows. When one machine can't hold your data or handle your write throughput, you shard. It's the only way to scale writes and storage beyond the limits of a single machine."*

---

### ❓ What Problem Does It Solve?
**The single-machine ceiling:**
- Maximum RAM on one server: ~24TB (and extremely expensive)
- Maximum write throughput: ~100K writes/second per server
- Maximum storage: ~100TB on one server practically

At internet scale, these limits are hit. You can't store Twitter's 500 million daily tweets on one server. You can't process Stripe's millions of transactions per second on one DB. Sharding breaks these limits by distributing across many servers.

### 🏭 Why Needed in Production?
- **Storage** — dataset bigger than any single machine's disk
- **Write throughput** — more writes per second than any single machine can handle
- **Query isolation** — a heavy analytics query on one shard doesn't slow down other shards
- **Geographic distribution** — put shards close to the users who need them
- **Independent scaling** — scale only the hot shards that need more resources

### 💥 What Happens Without It?
- Database grows beyond one server's RAM → all queries go to disk → 100x slower
- Write throughput saturates → writes queue → latency spikes → data loss under peak load
- One noisy analytics query blocks all other queries on the single server

### 🏢 Real Company Example
**WhatsApp** — 100 billion messages per day. One PostgreSQL server cannot store or handle this. WhatsApp shards its message data across thousands of database nodes — each shard responsible for a subset of user IDs. A message from User A to User B goes to the shard that owns User A's data.

---

### 📐 Sharding vs Replication

```
REPLICATION: Same data, multiple machines
  Primary: [User 1, 2, 3, 4, 5, 6]
  Replica: [User 1, 2, 3, 4, 5, 6]  ← identical copy
  → Solves: availability + read scaling
  → Doesn't solve: write scaling or storage limits

PARTITIONING: Different data, multiple machines
  Shard 1: [User 1, 2, 3]
  Shard 2: [User 4, 5, 6]  ← different subset
  → Solves: write scaling + storage limits
  → Doesn't solve: availability (if shard 1 dies, users 1-3 are inaccessible)

BOTH TOGETHER (Production Reality):
  Shard 1 Primary: [User 1, 2, 3] ── replicated to ──► Shard 1 Replica
  Shard 2 Primary: [User 4, 5, 6] ── replicated to ──► Shard 2 Replica
  → Solves everything: scale + availability + fault tolerance
```

---

## 8. Partition by Key (Range Partitioning)

### 🧠 Simple Definition
Range Partitioning assigns rows to shards based on a **continuous range of the partition key**. Shard 1 gets keys A–F, Shard 2 gets G–M, Shard 3 gets N–Z. Or by number: Shard 1 gets user_id 1–1M, Shard 2 gets 1M–2M.

> **Interview Line to Remember:**
> *"Range partitioning is like an encyclopedia — A-F on Shard 1, G-M on Shard 2, N-Z on Shard 3. Great for range queries ('all orders in June') because they hit one shard. Weakness: sequential keys create hotspots — all new users hit the same shard."*

---

### ❓ What Problem Does It Solve?
**Range queries** — "show all orders between Jan 1 and Jan 31" can be answered by querying a single shard (all January data is on Shard 1). No need to query all shards and merge results. This is called a **shard-local query** — fast and cheap.

### 🏭 Why Needed in Production?
- **Time-series data** — sensor readings, logs, financial ticks partitioned by date range
- **Alphabetical data** — user lookup by last name, product by category name
- **Range scans** — "all transactions between $100 and $500" on a value-partitioned table
- **Data archival** — old time ranges can be moved to cheaper storage

### 💥 Range Partitioning's Big Weakness — Sequential Hotspots:
```
Partition key: timestamp (current time)
Shard 1: Jan data
Shard 2: Feb data
Shard 3: Mar data  ← ALL new writes go HERE (it's March now)
Shard 4: Apr data  ← empty, idle

Problem: Shard 3 gets 100% of writes. Shards 1, 2, 4 are idle.
This is a write hotspot — defeats the purpose of sharding.
```

### 🏢 Real Company Example
**HBase (used by Facebook for messages)** — rows are partitioned by user_id range. User IDs 0–10M on Region Server 1, 10M–20M on Region Server 2, etc. Facebook reads all messages for a user from one region server — a single contiguous range scan. Efficient, predictable access pattern.

---

### 📐 Range Partitioning Visualization

```
Partition Key: order_date

Shard 1:  Jan 2025 orders  [jan_001, jan_002, jan_003 ... jan_999]
Shard 2:  Feb 2025 orders  [feb_001, feb_002, feb_003 ... feb_999]
Shard 3:  Mar 2025 orders  [mar_001, mar_002, mar_003 ... mar_999]

Query: "All orders in January"
  → Hits only Shard 1 ✅  (efficient, no cross-shard join)

Query: "All orders from Jan to Mar"
  → Hits Shards 1, 2, 3 (scatter-gather, but still predictable)

Query: "Single order by order_id=feb_042"
  → Compute range → Shard 2 → single shard lookup ✅
```

---

## 9. Partition by Hash

### 🧠 Simple Definition
Hash Partitioning takes the partition key, runs it through a **hash function**, and uses the result to determine which shard gets the data. `shard = hash(key) % number_of_shards`. The hash function distributes data **uniformly and randomly** across shards.

> **Interview Line to Remember:**
> *"Hash partitioning runs the key through a hash function to pick the shard — hash(user_id) % 10 gives shard 0-9. It distributes data evenly, eliminating hotspots. The tradeoff: range queries now scatter across all shards because adjacent keys hash to random shards."*

---

### ❓ What Problem Does It Solve?
Range partitioning's hotspot problem — sequential keys pile up on one shard. Hash partitioning **scrambles the key space** so similar keys end up on different shards. Even if you insert 1 million consecutive user IDs, they're distributed roughly evenly across all shards.

### 🏭 Why Needed in Production?
- **Even write distribution** — no shard gets more writes than others (no hotspots)
- **Even storage** — each shard holds approximately the same amount of data
- **Predictable performance** — all shards equally loaded; no single bottleneck
- **Default in most distributed DBs** — MongoDB, DynamoDB, Cassandra all use consistent hashing

### 💥 Hash Partitioning's Weakness — Range Query Scatter:
```
Range query: "All orders with user_id between 1000 and 2000"

Range partitioning: → Hit 1 shard (they're in the same range) ✅
Hash partitioning:
  hash(1000) % 10 = 4  → Shard 4
  hash(1001) % 10 = 7  → Shard 7
  hash(1002) % 10 = 1  → Shard 1
  ...every key in the range hashes to a different shard

  → Must query ALL 10 shards and merge results ❌ (scatter-gather)
  → Expensive and slow for range queries
```

### 🏢 Real Company Example
**Cassandra (Netflix, Instagram)** — partition key is hashed using **consistent hashing** on a ring. Each node owns a range of the hash ring. Data is distributed so evenly that Netflix can add nodes or remove nodes with minimal data movement — each new node takes over only 1/N of the ring from adjacent nodes.

---

### 📐 Consistent Hashing — The Production Upgrade

```
Regular Hash: shard = hash(key) % N
Problem: Add a shard (N becomes N+1) → almost ALL keys remap → massive data movement

Consistent Hashing: Keys and nodes placed on a circular ring
  Hash ring: 0 ─────────────────────────────── 360°

         N1(at 0°)
        /         \
   N4(270°)    N2(90°)
        \         /
         N3(180°)

  Key K1 hashes to 45° → assigned to next node clockwise → N2
  Key K2 hashes to 200° → assigned to N3
  Key K3 hashes to 300° → assigned to N4

Add node N5 at 135°:
  Only keys between 90° and 135° move from N3 to N5
  All other keys unaffected ✅
  Only 1/N of data moves (not all of it)

Used by: Cassandra, DynamoDB, Memcached, Akamai CDN
```

---

## 10. Secondary Index Partitioning

### 🧠 Simple Definition
A secondary index lets you query by a **non-partition key column** (e.g., partition by user_id but query by email or location). The challenge: when data is sharded, secondary indexes must either be **scattered across all shards** (local indexes) or **maintained globally on special nodes** (global indexes).

> **Interview Line to Remember:**
> *"Secondary indexes in sharded systems are hard because the data is spread across shards. Local index: each shard indexes its own data — fast writes, scatter-gather reads. Global index: one index covers all shards — fast reads, slower writes. Every sharded DB must choose one."*

---

### ❓ What Problem Does It Solve?
You shard orders by `order_id`. A customer wants "all orders by user_id=42". Without a secondary index, you'd scan every shard looking for user 42's orders. Secondary indexes make non-partition-key queries efficient.

### 🏭 Why Needed in Production?
- Users query on multiple dimensions — by user, by product, by date, by status
- Partition key is chosen for write distribution; queries come from all directions
- Without secondary indexes, every non-partition-key query is a full cluster scan

### 📐 Two Approaches

**Approach 1: Local (Scatter-Gather) Index**
```
Shard 1 owns orders 1-1000:         Shard 2 owns orders 1001-2000:
  Data: order 1, 2, ... 1000          Data: order 1001, 1002, ... 2000
  Local index by user_id:             Local index by user_id:
    user_42 → [order_5, order_99]       user_42 → [order_1050, order_1200]
    user_17 → [order_12, order_44]      user_17 → [order_1100]

Query: "All orders for user_42"
  → Must ask BOTH shards (scatter)
  → Merge results: [5, 99, 1050, 1200] (gather)
  → Called: scatter-gather query

Write: Fast — only update local shard's index
Read:  Slow — must query all shards
Used by: MongoDB secondary indexes
```

**Approach 2: Global Index (Term-Partitioned)**
```
Separate global index shards — indexed by the secondary key:

Index Shard A (user_ids 1-500):
  user_42 → [order_5, order_99, order_1050, order_1200]  ← all shards

Index Shard B (user_ids 501-1000):
  user_800 → [order_77, order_512]

Query: "All orders for user_42"
  → Hit Index Shard A (single node) → get all order_ids → fetch orders
  → Fast reads ✅

Write: Slow — must update the global index shard too (2 writes per insert)
       Async update risks stale index
Used by: DynamoDB Global Secondary Index (GSI)
```

### 🏢 Real Company Example
**DynamoDB Global Secondary Index (GSI)** — used by Amazon's own systems. Table partitioned by `product_id`, but you also query by `seller_id`. The GSI maintains a global index on `seller_id` so "all products by this seller" is a fast single-node lookup rather than a scatter-gather across all partitions.

---

## 11. Hotspots and Challenges

### 🧠 Simple Definition
A **Hotspot** is a shard that receives **disproportionately more traffic** than other shards — becoming a bottleneck while other shards sit idle. It defeats the entire purpose of sharding.

> **Interview Line to Remember:**
> *"A hotspot is when one shard gets all the traffic — like everyone going to the same checkout lane. It bottlenecks that one machine while others are idle. Caused by bad partition key choice or inherently skewed data (celebrity problem). Solved by key salting, sub-partitioning, or smarter key design."*

---

### ❓ What Problem Does It Solve?
Hotspots are the #1 failure mode of sharding strategies. Understanding them is what separates good system design from great system design.

### 🏭 Why Needed in Production?
Every partitioning decision must account for hotspots — they cause:
- One shard at 100% CPU while others are at 10% → cascading failure on that shard
- Latency spikes for all users whose data is on the hot shard
- Wasted capacity on idle shards

---

### 📐 Common Hotspot Causes

**Cause 1: Sequential Key Hotspot (Range Partitioning)**
```
Bad partition key: created_at (timestamp)
All new records are written to the "current time" shard.

  Shard Jan: 100 writes today (all old data, nobody writing)
  Shard Feb: 200 writes today (old data)
  Shard Jun: 9,999,700 writes today ← ALL new writes GO HERE 🔥

Fix: Use hash partitioning instead of range on time.
     Or: prepend a random prefix to the key (key salting).
```

**Cause 2: Celebrity / Skewed Data Hotspot (Hash Partitioning)**
```
hash("user_id_42") % 10 = 3 → Shard 3

User 42 = Cristiano Ronaldo (Instagram) — 600M followers
Every time CR7 posts:
  → 600M read requests for his latest post
  → All 600M reads → hash(cr7_user_id) → Shard 3
  → Shard 3 on fire 🔥, Shards 1,2,4-9 idle

Fix options:
  1. Celebrity detection: cache CR7's data separately in Redis
  2. Key salting: store CR7's data as user_42_0, user_42_1... user_42_99
     → 100 different hashes → 100 different shards → 600M reads spread out
  3. Read replicas specifically for hot keys
```

**Cause 3: Uneven Data Distribution**
```
Partition by country:
  USA shard:    300M users  🔥
  India shard:  1.4B users  🔥🔥🔥
  Monaco shard: 36K users   (idle)

Fix: Partition by user_id hash, not by country.
     Use GEO routing at the application layer for locality.
```

---

### 📐 Key Salting — The Hotspot Fix

```
Problem: User 42 (celebrity) causes hotspot on Shard 3

Key Salting:
  Instead of key = "user_42"
  Use keys: "user_42_00" through "user_42_99"  (100 virtual keys)

  hash("user_42_00") % 10 = 3
  hash("user_42_01") % 10 = 7
  hash("user_42_02") % 10 = 1
  ... (all different shards)

Write: Pick a random salt (0-99), write to that shard
       Store {user: 42, salt: 37, data: ...} on Shard that 42_37 hashes to

Read: Must read from ALL 100 salted keys across multiple shards and merge
      (Trade: reads become scatter-gather, but at least no hotspot)

Used by: Twitter's approach for celebrity tweets
```

---

### 📐 Sharding Challenges Summary

| Challenge                 | Description                                      | Solution                                      |
|--------------------------|--------------------------------------------------|-----------------------------------------------|
| **Hotspots**             | One shard overloaded, others idle                | Better key choice, salting, caching hot keys  |
| **Cross-shard joins**    | JOIN across two shards = two DB calls + merge    | Denormalize, co-locate related data           |
| **Cross-shard transactions** | ACID across 2 shards needs 2PC (slow, complex) | Avoid or use saga pattern                  |
| **Rebalancing**          | Adding shards requires moving data               | Consistent hashing minimizes movement         |
| **Secondary indexes**    | Non-partition queries scatter across all shards  | Global index or denormalize for query patterns|
| **Schema changes**       | ALTER TABLE on billions of rows across shards    | Online schema change tools (gh-ost, pt-osc)   |

---

## 12. Replication vs Partitioning vs Both

### 📐 The Definitive Comparison

| Feature                | Replication                         | Partitioning (Sharding)               |
|-----------------------|--------------------------------------|---------------------------------------|
| **What it does**      | Copies same data to multiple nodes   | Splits different data across nodes    |
| **Solves**            | Availability + read scaling          | Write scaling + storage limits        |
| **Data per node**     | All data on every node               | Subset of data per node               |
| **Node fails**        | Other replicas serve traffic         | That shard's data is unavailable      |
| **Write throughput**  | Limited by primary node              | Scales linearly with shards           |
| **Read throughput**   | Scales linearly with replicas        | Scales linearly with shards           |
| **Complexity**        | Moderate (lag, failover)             | High (routing, joins, hotspots)       |

---

### 📐 Production Reality — Use Both

```
Production System (e.g. Cassandra, DynamoDB, MongoDB):

Data is sharded into 4 partitions + each partition is replicated 3 times:

Shard 1 (users 1-25%):
  Primary_1A  ◄──── writes ──── Router
  Replica_1B  (sync/async copy of 1A)
  Replica_1C  (sync/async copy of 1A)

Shard 2 (users 25-50%):
  Primary_2A
  Replica_2B
  Replica_2C

Shard 3 (users 50-75%):
  Primary_3A
  Replica_3B
  Replica_3C

Shard 4 (users 75-100%):
  Primary_4A
  Replica_4B
  Replica_4C

Total: 4 shards × 3 replicas = 12 nodes
Sharding gives write scale. Replication gives fault tolerance.
```

---

## 13. Quick Revision Cheatsheet

```
REPLICATION
────────────────────────────────────────────────────────────────────
Why needed    = Availability (no data loss on failure) + Read scaling
Single Leader = One primary accepts all writes. Replicas serve reads.
               Simple, no conflicts. Weakness: write bottleneck, failover lag.
Multi Leader  = Multiple primaries, each accepts writes. Async sync between them.
               Solves: cross-DC write latency. Hard part: write conflicts.
               Conflict resolve: LWW (last write wins), CRDT, user resolution.
Leaderless    = No master. Write to N nodes, read from N nodes, quorum decides.
               Highest availability. Used by Cassandra, DynamoDB, Riak.
               Self-heals via Read Repair.

SYNC vs ASYNC
──────────────
Sync  = Leader waits for replica ACK. Zero data loss. Slower writes.
Async = Leader confirms immediately, replicates later. Fast. Risk: data loss.
Semi  = Wait for 1 replica ACK, rest async. Best of both. (Production default)

QUORUM
──────
Formula: W + R > N → guaranteed read of latest write
N=5, W=3, R=3 → Strong consistency, tolerates 2 node failures
N=3, W=2, R=2 → DynamoDB default. Tolerates 1 failure.
W+R ≤ N       → Eventual consistency only

PARTITIONING (SHARDING)
────────────────────────────────────────────────────────────────────
Why needed       = Dataset too big / writes too fast for one machine
Range Partition  = Shard by key range (A-F, G-M, N-Z or Jan/Feb/Mar)
                   Good: range queries hit 1 shard. Bad: sequential hotspots.
Hash Partition   = shard = hash(key) % N. Even distribution.
                   Good: no hotspots. Bad: range queries scatter all shards.
Consistent Hash  = Keys+nodes on a ring. Add node → only 1/N keys move.
                   Used by Cassandra, DynamoDB, Akamai CDN.

SECONDARY INDEXES
──────────────────
Local (scatter-gather) = Each shard indexes own data. Fast writes, slow reads.
Global (term-partition) = Centralized index across all shards. Fast reads, slow writes.

HOTSPOTS
──────────
Sequential keys  → range hotspot → use hash or add random prefix
Celebrity data   → hash hotspot → key salting or dedicated cache
Skewed geography → country shard → use user_id hash instead

FIX: Key Salting = append random suffix (0-99) to hot key
     Spreads one key across 100 virtual keys → 100 shards
     Trade: writes fast, reads scatter-gather

REPLICATION vs PARTITIONING
─────────────────────────────
Replication = same data, many machines → availability + read scale
Partitioning = different data, many machines → write scale + storage
Production  = USE BOTH: shard data + replicate each shard
```

---

*Made for interview prep — understand the concept, remember the line, nail the answer.*